<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-06-rag-systems/lesson-6.11-rag-evaluation-and-ragas/notebooks/exercises-6.11.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 6.11 — RAG Evaluation & RAGAS
Netsetos GenAI Course v2.0 | Module 6

RAGAS metrics, DeepEval, retrieval metrics, synthetic test generation, observability, human evaluation


In [ ]:
# pip install ragas deepeval langchain-openai -q
print('Ready for RAG Evaluation')


## Ex 1: Faithfulness From Scratch


In [ ]:
def compute_faithfulness(answer, context):
    # Step 1: Extract claims from answer
    claims = answer.split('. ')
    # Step 2: Check each claim against context
    supported = sum(1 for c in claims if any(k in context.lower() for k in c.lower().split()[:3]))
    return supported / len(claims) if claims else 0

context = 'The company reported $4.2M revenue in Q3 2024, up 15% from Q2.'
answer = 'Revenue was $4.2M in Q3. Growth was 15%. The CEO was optimistic.'
print(f'Faithfulness: {compute_faithfulness(answer, context):.2f}')
print('  2/3 claims supported (CEO optimism = hallucination)')


## Ex 2: RAGAS Library — 4 Metrics


In [ ]:
# from ragas import evaluate
# from ragas.metrics import faithfulness, answer_relevancy
# from ragas.metrics import context_precision, context_recall
# result = evaluate(dataset, metrics=[faithfulness, answer_relevancy,
#     context_precision, context_recall])
# print(result)

print('RAGAS 4 core metrics:')
for m, desc in [
    ('Faithfulness','claims_supported / total_claims (anti-hallucination)'),
    ('Answer Relevancy','does answer address the question?'),
    ('Context Precision','are top-ranked docs more relevant?'),
    ('Context Recall','did we retrieve ALL relevant docs?'),
]: print(f'  {m:22s}: {desc}')
print('\nProduction gate: faithfulness >= 0.80')


## Ex 3: DeepEval pytest Integration


In [ ]:
# from deepeval import assert_test
# from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
# from deepeval.test_case import LLMTestCase
#
# def test_rag():
#     test_case = LLMTestCase(
#         input='What is the refund policy?',
#         actual_output='Refunds within 30 days with receipt.',
#         retrieval_context=['Refund policy: 30 days, receipt required.']
#     )
#     assert_test(test_case, [
#         FaithfulnessMetric(threshold=0.8),
#         AnswerRelevancyMetric(threshold=0.7),
#     ])

print('DeepEval unique metrics beyond RAGAS:')
for m in ['GEval (custom criteria, 10M+/mo)',
    'HallucinationMetric (context as ground truth)',
    'BiasMetric (gender, racial, political)',
    'ToxicityMetric','PIILeakageMetric',
    'ConversationalTestCase (7 multi-turn metrics)',
    'DAG (deterministic decision-tree eval)']:
    print(f'  ✅ {m}')


## Ex 4: Retrieval Metrics — NDCG, MRR


In [ ]:
import numpy as np

def ndcg_at_k(relevances, k):
    rel = np.array(relevances[:k])
    dcg = np.sum(rel / np.log2(np.arange(2, len(rel) + 2)))
    ideal = np.sum(sorted(rel, reverse=True) / np.log2(np.arange(2, len(rel) + 2)))
    return dcg / ideal if ideal > 0 else 0.0

def mrr(rankings):
    return np.mean([1/r for r in rankings])

# Example: 5 retrieved docs with graded relevance (3=highly, 0=irrelevant)
rels = [3, 0, 2, 0, 1]  # best doc at rank 1, noise at 2 and 4
print(f'NDCG@5: {ndcg_at_k(rels, 5):.3f}')
print(f'MRR (first relevant at rank 1): {mrr([1, 3, 1, 2]):.3f}')
print(f'\nTarget: NDCG@10 >= 0.8 for production RAG')


## Ex 5: RAGAS TestsetGenerator


In [ ]:
# from ragas.testset import TestsetGenerator
# from ragas.testset.synthesizers import (
#     SingleHopSpecificQuerySynthesizer,
#     MultiHopAbstractQuerySynthesizer,
# )
# from ragas.llms import llm_factory
#
# llm = llm_factory('gpt-4o')
# generator = TestsetGenerator(llm=llm)
# dataset = generator.generate_with_langchain_docs(docs, testset_size=100)

print('RAGAS v0.2+ TestsetGenerator pipeline:')
print('  1. KG construction from documents')
print('  2. Graph enrichment (summaries, themes, entities)')
print('  3. Query synthesis via synthesizers')
print()
print('Synthesizers (replace old evolution types):')
print('  SingleHopSpecificQuerySynthesizer → replaces "simple"')
print('  MultiHopAbstractQuerySynthesizer → replaces "multi_context"')
print('  MultiHopSpecificQuerySynthesizer → new')
print()
print('Sizing: 50-100 golden + 200-500 synthetic')
print('Cost: ~$0.001-0.003/test case with GPT-4o-mini judge')


## Ex 6: Component Isolation — Diagnose Bad RAG


In [ ]:
print('Diagnostic framework:')
print()
print('Symptom: Bad answers')
print('  Step 1: Check faithfulness (generation quality)')
print('    High (>0.8) → LLM is grounded, problem is upstream')
print('    Low (<0.7) → LLM is hallucinating, fix generation')
print()
print('  Step 2: Check NDCG@10 (retrieval quality)')
print('    High (>0.8) → retrieval is good')
print('    Low (<0.6) → retrieval failing, fix embedding/chunking')
print()
print('  Step 3: Check Context Precision + Recall')
print('    High Precision + Low Recall → too selective, increase top-k')
print('    Low Precision + High Recall → too noisy, add reranker')
print()
print('Common trap: High faithfulness + bad answers = retrieval failure')
print('The LLM faithfully generates from WRONG context')


## Ex 7: Observability Setup


In [ ]:
print('Production monitoring checklist:')
print()
print('Instrument with Langfuse/Phoenix:')
print('  Track per-query: latency, tokens, cost, faithfulness')
print('  Run faithfulness on 10-20% of production traffic')
print('  Rolling 7-day faithfulness average')
print()
print('Alert thresholds:')
for alert in [
    'Error rate > 5% over 5 min',
    'P95 latency > 2x baseline for 10+ min',
    'Faithfulness drop > 10% from rolling average',
    'Daily cost > 150% of 7-day average',
    '"I don\'t know" rate spike > 2x',
]: print(f'  🚨 {alert}')
print()
print('Cost at 50M spans/month:')
for p, c in [('LangSmith','$25-50K'),('Langfuse','~$4K'),
    ('Phoenix self-hosted','~$3-8K')]: print(f'  {p:25s}: {c}')


## Ex 8: Human Evaluation Pipeline


In [ ]:
print('Hybrid evaluation pipeline:')
print()
print('Tier 1: LLM-as-judge on ALL outputs (bulk screening)')
print('  RAGAS faithfulness + answer relevancy')
print()
print('Tier 2: Threshold routing (faithfulness < 0.7 → human review)')
print('  Route low-scoring outputs to expert reviewers')
print()
print('Tier 3: Random spot-check (5-10% of PASSING outputs)')
print('  Catches LLM-as-judge blind spots')
print()
print('Tier 4: Feedback loop')
print('  Human judgments recalibrate judge instructions')
print('  Even 10-20 reviews/week drives major quality gains')
print()
print('Inter-annotator agreement:')
print('  Cohen\'s Kappa (2 raters): target κ >= 0.61')
print('  Krippendorff\'s Alpha (3+ raters): target α >= 0.667')


---
## Recap
RAG evaluation is a SYSTEM, not a metric. Four RAGAS metrics (faithfulness, answer relevancy, context precision, context recall) provide the nucleus — gate on faithfulness ≥ 0.80. DeepEval adds pytest-native CI/CD, GEval custom metrics, safety suite, and 7 multi-turn metrics. Retrieval metrics (NDCG@k, MRR) isolate retrieval from generation — high faithfulness + bad answers = retrieval failure. Synthetic test data: RAGAS v0.2+ TestsetGenerator (KG→enrichment→synthesis), 50-100 golden + 200-500 synthetic. Observability: Langfuse (~$4K at scale) vs LangSmith (~$25-50K). Component evaluation + ablation testing prove which improvements actually work. Human evaluation: 12 LLM-as-judge biases (CALM ICLR 2025), Cohen's Kappa ≥ 0.61, hybrid pipeline (automated → threshold → spot-check → calibrate). India: DeepRAG +23% Hindi retrieval, MIRAGE-Bench 18 languages, LE-BTL Indian legal benchmark. This completes Module 6: RAG.
